In [1]:
# VARIATIONAL AUTOENCODER (VAE) ON MNIST -- PYTORCH VERSION
# ------------------------------------------------------------


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os # Import the os module

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the output directory if it doesn't exist
output_dir = "/mnt/user-data/outputs/"
os.makedirs(output_dir, exist_ok=True)

# -------------------------------------------------
# 1. Data: MNIST handwritten digits (via torchvision)
# -------------------------------------------------
original_dim = 784
latent_dim = 2   # 2D so we can easily visualize/generate & plot the latent space
batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor(),                       # scales pixels to [0, 1]
    transforms.Lambda(lambda x: x.view(-1)),      # flatten 28x28 -> 784
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# -------------------------------------------------
# 2. VAE model: encoder -> reparameterize -> decoder
# -------------------------------------------------
class VAE(nn.Module):
    def __init__(self, original_dim=784, latent_dim=2):
        super().__init__()
        # Encoder
        self.enc_fc1 = nn.Linear(original_dim, 256)
        self.enc_fc2 = nn.Linear(256, 64)
        self.fc_mean = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)

        # Decoder
        self.dec_fc1 = nn.Linear(latent_dim, 64)
        self.dec_fc2 = nn.Linear(64, 256)
        self.dec_out = nn.Linear(256, original_dim)

    def encode(self, x):
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        return self.fc_mean(h), self.fc_logvar(h)

    def reparameterize(self, z_mean, z_logvar):
        # The "reparameterization trick": sample z = mean + std * epsilon
        # so gradients can flow through the sampling step.
        std = torch.exp(0.5 * z_logvar)
        epsilon = torch.randn_like(std)
        return z_mean + std * epsilon

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        return torch.sigmoid(self.dec_out(h))

    def forward(self, x):
        z_mean, z_logvar = self.encode(x)
        z = self.reparameterize(z_mean, z_logvar)
        reconstruction = self.decode(z)
        return reconstruction, z_mean, z_logvar

# -------------------------------------------------
# 3. Loss = reconstruction loss + KL divergence
# -------------------------------------------------
def vae_loss(reconstruction, x, z_mean, z_logvar):
    # Reconstruction loss: binary cross-entropy summed over pixels
    recon_loss = F.binary_cross_entropy(reconstruction, x, reduction="sum")
    # KL divergence: keeps the latent distribution close to N(0, 1)
    kl_loss = -0.5 * torch.sum(1 + z_logvar - z_mean.pow(2) - z_logvar.exp())
    return recon_loss + kl_loss, recon_loss, kl_loss

# -------------------------------------------------
# 4. Train
# -------------------------------------------------
model = VAE(original_dim, latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 15

for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_recon, total_kl, n_samples = 0.0, 0.0, 0.0, 0

    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()

        reconstruction, z_mean, z_logvar = model(x)
        loss, recon_loss, kl_loss = vae_loss(reconstruction, x, z_mean, z_logvar)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        n_samples += x.size(0)

    print(
        f"Epoch {epoch:2d}/{epochs} | "
        f"loss: {total_loss / n_samples:.4f} | "
        f"recon: {total_recon / n_samples:.4f} | "
        f"kl: {total_kl / n_samples:.4f}"
    )

# -------------------------------------------------
# 5. Generate new digits by sampling a grid over latent space
# -------------------------------------------------
model.eval()
n = 15
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)[::-1]

with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            x_decoded = model.decode(z_sample).cpu().numpy()
            digit = x_decoded[0].reshape(digit_size, digit_size)
            figure[
                i * digit_size : (i + 1) * digit_size,
                j * digit_size : (j + 1) * digit_size,
            ] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap="Greys_r")
plt.axis("off")
plt.title("Digits generated by sampling the VAE latent space")
plt.savefig(os.path.join(output_dir, "vae_pytorch_generated_digits.png"), dpi=150, bbox_inches="tight")
plt.close()

# -------------------------------------------------
# 6. Visualize how real digits cluster in latent space (colored by true label)
# -------------------------------------------------
all_z_mean = []
all_labels = []

with torch.no_grad():
    for x, y in train_loader:
        x = x.to(device)
        z_mean, _ = model.encode(x)
        all_z_mean.append(z_mean.cpu().numpy())
        all_labels.append(y.numpy())

all_z_mean = np.concatenate(all_z_mean, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(all_z_mean[:, 0], all_z_mean[:, 1], c=all_labels, cmap="tab10", s=2)
plt.colorbar(scatter, label="Digit label")
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.title("MNIST digits mapped into 2D latent space")
plt.savefig(os.path.join(output_dir, "vae_pytorch_latent_space.png"), dpi=150, bbox_inches="tight")
plt.close()

print("Done. Saved: vae_pytorch_generated_digits.png and vae_pytorch_latent_space.png")

100%|██████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:14<00:00, 672kB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 100kB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:02<00:00, 665kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 7.12MB/s]


Epoch  1/15 | loss: 195.2053 | recon: 191.2980 | kl: 3.9073
Epoch  2/15 | loss: 166.9705 | recon: 161.9646 | kl: 5.0059
Epoch  3/15 | loss: 161.4852 | recon: 156.1812 | kl: 5.3040
Epoch  4/15 | loss: 158.3135 | recon: 152.8702 | kl: 5.4432
Epoch  5/15 | loss: 156.1249 | recon: 150.5936 | kl: 5.5314
Epoch  6/15 | loss: 154.3107 | recon: 148.6785 | kl: 5.6322
Epoch  7/15 | loss: 152.7938 | recon: 147.0695 | kl: 5.7243
Epoch  8/15 | loss: 151.4811 | recon: 145.6411 | kl: 5.8401
Epoch  9/15 | loss: 150.4403 | recon: 144.5310 | kl: 5.9093
Epoch 10/15 | loss: 149.3653 | recon: 143.3835 | kl: 5.9818
Epoch 11/15 | loss: 148.6663 | recon: 142.6409 | kl: 6.0253
Epoch 12/15 | loss: 147.8072 | recon: 141.7226 | kl: 6.0846
Epoch 13/15 | loss: 147.2558 | recon: 141.1132 | kl: 6.1426
Epoch 14/15 | loss: 146.5267 | recon: 140.3347 | kl: 6.1920
Epoch 15/15 | loss: 146.0206 | recon: 139.8023 | kl: 6.2184
Done. Saved: vae_pytorch_generated_digits.png and vae_pytorch_latent_space.png
